<a href="https://colab.research.google.com/github/mahendrapd/GRST/blob/main/RareAttacksRST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [51]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [52]:
data = pd.read_parquet('/content/drive/MyDrive/dataset/training.parquet')

In [53]:
cols=len(data.columns)
nfeatures = cols-1
X = data.iloc[:,0:nfeatures]
y = data.iloc[:,-1]
print(len(data),cols)

347122 55


In [54]:
#targetclass='Infiltration'
D=['Bruteforce', 'Botnet', 'Portscan', 'Webattack']


In [55]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [56]:
def RSTClassifier(X,y):
    #print(targetclass)
    dc=pd.Series(y)
    cm=dc.groupby(dc).apply(lambda px: px.index.tolist()).to_dict() #index of targetclass
    #D=set(cm[targetclass])
    D1=set(cm['Bruteforce'])
    D2=set(cm['Botnet'])
    D3=set(cm['Portscan'])
    D4=set(cm['Webattack'])
    #D5=set(cm['DoS'])
    #D6=set(cm['Infiltration'])
    #D7=set(cm['DDoS'])
    #D=D1.union(D2).union(D3).union(D4).union(D5).union(D6).union(D7)
    D=D1.union(D2).union(D3).union(D4)
    print(D)
    nfeatures=len(X.columns)
    #print(cm)
    purity={}
    headers = X.keys().tolist()

    for j in range(nfeatures):
        header=headers[j]
        hdata={}
        #print("Columns:",j,header)
        dp=pd.Series(X.iloc[:,j])
        md=dp.groupby(dp).apply(lambda px: px.index.tolist()).to_dict()
        K=md.keys()
        #print(K)
        for z in K:
            common_elements=set(md[z]) & D
            val=round(len(common_elements)/len(md[z]),3)
            #print(z,len(md[z]),len(common_elements),val)
            hdata[z]=val
        purity[header]=hdata
    #print(purity['urgent'][0])
    return purity

In [57]:
pur=RSTClassifier(X_train,y_train)

{110595, 110597, 54282, 110604, 110605, 94220, 54287, 310791, 291864, 291865, 97306, 291866, 110623, 110624, 124961, 110625, 124962, 110628, 110629, 124966, 124967, 124965, 124969, 342049, 124971, 124972, 124973, 111661, 124975, 124976, 124977, 124978, 124979, 84015, 111669, 84016, 53296, 53299, 53303, 53304, 53305, 53306, 53307, 124990, 124991, 124992, 124993, 124994, 124995, 124996, 53311, 124998, 124999, 310801, 125001, 341061, 125003, 125004, 340045, 125007, 125008, 342096, 125010, 125012, 125014, 125019, 94320, 83056, 88178, 82035, 112761, 84093, 84094, 53373, 82068, 94358, 298145, 298147, 94374, 298150, 298151, 94391, 299196, 85184, 84166, 298189, 341561, 110805, 84183, 342581, 84194, 341565, 298220, 55536, 55538, 342586, 122101, 55541, 85241, 83195, 82172, 342267, 84235, 84236, 84237, 54543, 110865, 82198, 342295, 54552, 342297, 342299, 298269, 291305, 342301, 111910, 295209, 85290, 295210, 295212, 85293, 295213, 298287, 85296, 295218, 298290, 85300, 85301, 294197, 85303, 291311

In [58]:
#print(pur)

In [59]:
def RST_AmbigiousSample(X,pur):
    headers = X.keys().tolist()
    print(headers,len(headers),len(X))
    ambigioussamples={}
    for i in range(len(X)):
        flag=0
        sumpurity=0
        for j in range(len(headers)):
            key=X.iloc[i][headers[j]]
            if (pur[headers[j]][key] == 1 or pur[headers[j]][key] == 0):
                flag=1
                break
            else:
                sumpurity+=pur[headers[j]][key]
        if (flag == 0):
            Avgpurity=round(sumpurity/len(headers),3)
            ambigioussamples[i]=Avgpurity
    return ambigioussamples

In [60]:
AmbSample=RST_AmbigiousSample(X_train,pur)

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle 

In [61]:
print(len(AmbSample))

5918


In [62]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [63]:
def RST_Gradient(y,sample,epoch=300):
    Theta=0.5
    LRate=0.01
    pc=0
    for i in range(epoch):
        correctcount=0
        val=0
        Th=Theta
        L=0

        for key, value in sample.items():
            if (value >= Theta and y.iloc[key] in D):
                correctcount+=1
            elif (value < Theta and y.iloc[key] not in D):
                correctcount+=1
            else:
                val+=value
        ictotal=(len(sample)-correctcount)
        try:
          p=correctcount/len(sample)
          #L=-(Th*math.log2(p)+(1-Th)*math.log2(1-p))
          #L=-((Th/p)-((1-Th)/(1-p)))
          #L=np.tanh(L)
          #Theta=abs(round((Theta-LRate*L),3))
          #print(L)
          if (pc<=correctcount):
            Theta=abs(round(Theta-LRate*(sigmoid((val/ictotal))),3))
            pc=correctcount
          #Theta=round(Theta-LRate*(0-(val/ictotal))*(val/ictotal)*(1-(val/ictotal)),3)
        except:
          Theta=Theta
        print(i, correctcount, ictotal, len(sample), val, Theta)

    return Theta

In [64]:
Th=RST_Gradient(y_train,AmbSample,300)

0 5575 343 5918 1.6929999999999958 0.495
1 5575 343 5918 1.6929999999999958 0.49
2 5575 343 5918 1.6929999999999958 0.485
3 5575 343 5918 1.6929999999999958 0.48
4 5575 343 5918 1.6929999999999958 0.475
5 5575 343 5918 1.6929999999999958 0.47
6 5575 343 5918 1.6929999999999958 0.465
7 5575 343 5918 1.6929999999999958 0.46
8 5575 343 5918 1.6929999999999958 0.455
9 5575 343 5918 1.6929999999999958 0.45
10 5575 343 5918 1.6929999999999958 0.445
11 5575 343 5918 1.6929999999999958 0.44
12 5575 343 5918 1.6929999999999958 0.435
13 5575 343 5918 1.6929999999999958 0.43
14 5575 343 5918 1.6929999999999958 0.425
15 5575 343 5918 1.6929999999999958 0.42
16 5575 343 5918 1.6929999999999958 0.415
17 5575 343 5918 1.6929999999999958 0.41
18 5575 343 5918 1.6929999999999958 0.405
19 5575 343 5918 1.6929999999999958 0.4
20 5575 343 5918 1.6929999999999958 0.395
21 5575 343 5918 1.6929999999999958 0.39
22 5575 343 5918 1.6929999999999958 0.385
23 5575 343 5918 1.6929999999999958 0.38
24 5575 343 591

In [65]:
def RST_Validation(X,y,pur,Theta):
    TP=0
    TN=0
    FP=0
    FN=0
    headers = X.keys().tolist()
    for i in range(len(X)):
        val=0
        count=0
        pval=0

        flag=0
        for j in range(len(headers)):
            #KYS=list(pur[headers[j]].keys())
            #dflag=0
            key=X.iloc[i][headers[j]]
            try:
              pval=pur[headers[j]][key]
            except KeyError:
              pval=Theta

            if ((pval == 1 or pval == 0)):
                flag=1
                break
            else:
                val+=pval
                count+=1

        if (flag == 1):
            if (pval==1 and y.iloc[i]in D):
                TP+=1
            elif (pval==1 and y.iloc[i] not in D):
                FN+=1
            elif (pval==0 and y.iloc[i] not in D):
                TN+=1
            else:
                FP+=1
        else:
            Aval=round((val/count),3)
            if (Aval>=Theta and y.iloc[i] in D):
                TP+=1
            elif (Aval<Theta and y.iloc[i] not in D):
                TN+=1
            elif (Aval>=Theta and y.iloc[i] not in D):
                FN+=1
            else:
                FP+=1
    try:
      Recall=round(TP/(TP+FN),4)
    except:
      Recall=0
    try:
      FPR=round(FP/(TN+FP),4)
    except:
      FPR=0
    try:
      Precision=round(TP/(TP+FP),4)
    except:
      Precision=0
    try:
      Accuracy=round((TP+TN)/(TP+TN+FP+FN),4)
    except:
      Accuracy=0
    try:
      FScore=round(2*Recall*Precision/(Recall+Precision),4)
    except:
      FScore=0
    print("TP=",TP,"\tTN=",TN,"\tFP=",FP,"\tFN=",FN)
    return Recall, FPR, Precision, FScore, Accuracy

In [66]:
Result=RST_Validation(X_train,y_train,pur,Th)
print(Result)

TP= 18 	TN= 259847 	FP= 475 	FN= 1
(0.9474, 0.0018, 0.0365, 0.0703, 0.9982)


In [67]:
ResultTest = RST_Validation(X_test,y_test,pur,Th)
print(ResultTest)

TP= 2 	TN= 86637 	FP= 141 	FN= 1
(0.6667, 0.0016, 0.014, 0.0274, 0.9984)


In [68]:
def CountPatterns(pur):
  value=0
  for outer_key, outer_value in pur.items():
    acount=0
    ncount=0
    for inner_key, inner_value in outer_value.items():
      if (inner_value == 0):
        acount=acount+1
      if (inner_value == 1):
        ncount=ncount+1
    print(f"{outer_key}: {acount}, {ncount}, {len(outer_value.items())}")

In [69]:
CountPatterns(pur)

Flow Duration: 10, 0, 11
Total Fwd Packets: 78, 0, 79
Total Backward Packets: 45, 0, 46
Fwd Packets Length Total: 11, 0, 12
Bwd Packets Length Total: 36, 0, 37
Fwd Packet Length Max: 30, 0, 43
Fwd Packet Length Mean: 18, 7, 43
Fwd Packet Length Std: 17, 1, 40
Bwd Packet Length Max: 23, 0, 32
Bwd Packet Length Mean: 13, 0, 17
Bwd Packet Length Std: 23, 0, 34
Flow Bytes/s: 72, 0, 93
Flow Packets/s: 22, 0, 39
Flow IAT Mean: 4, 0, 5
Flow IAT Std: 10, 0, 11
Flow IAT Max: 11, 0, 12
Flow IAT Min: 7, 0, 8
Fwd IAT Total: 10, 0, 11
Fwd IAT Mean: 4, 0, 5
Fwd IAT Std: 10, 0, 11
Fwd IAT Max: 11, 0, 12
Fwd IAT Min: 7, 0, 8
Bwd IAT Total: 56, 0, 101
Bwd IAT Mean: 77, 0, 101
Bwd IAT Std: 95, 0, 101
Bwd IAT Max: 70, 0, 101
Bwd IAT Min: 77, 0, 101
Fwd Header Length: 89, 0, 91
Bwd Header Length: 8, 0, 9
Fwd Packets/s: 30, 0, 46
Bwd Packets/s: 19, 0, 33
Packet Length Max: 32, 0, 49
Packet Length Mean: 13, 0, 23
Packet Length Std: 14, 0, 31
Packet Length Variance: 7, 0, 12
Avg Packet Size: 14, 0, 24
Avg Fw